# DFS Lineup Optimizer — Core Algorithm

This notebook documents and implements the **integer linear programming (ILP) optimizer** used to
select the highest-projected DraftKings NFL Classic lineup within the $50,000 salary cap.

It is intended as a **library / reference notebook** — run it to understand how each component
works, or `%run` it from `dfs_pipeline.ipynb` to load all functions into memory.


## 1 · Imports & Setup

We rely on three external libraries beyond the standard library:

| Library | Role |
|---------|------|
| `pandas` | Player pool table manipulation |
| `pulp` | Open-source ILP solver (CBC backend) |
| `difflib` | Fuzzy name matching between DK and our model |

`pathlib.Path` is used throughout so the notebook works regardless of working directory.


In [ ]:
import io
import re
import difflib
from pathlib import Path

import pandas as pd
import pulp

# Resolve the fantasy_projections directory relative to this file's location.
# Works whether you run from the project root or from inside fantasy/dfs/.
_HERE = Path().resolve()
PROJ_DIR = next(
    (p for p in [
        _HERE / "fantasy_projections",
        _HERE.parent / "fantasy_projections",
        _HERE.parent.parent / "fantasy" / "fantasy_projections",
    ] if p.exists()),
    _HERE / "fantasy_projections",
)
if not PROJ_DIR.exists():
    raise RuntimeError(f"Could not find fantasy_projections directory. Searched multiple locations. PROJ_DIR={PROJ_DIR}")

BUDGET = 50_000

print(f"Projections directory: {PROJ_DIR}")
print(f"Exists: {PROJ_DIR.exists()}")


## 2 · Name Normalisation

DraftKings and nflreadpy use slightly different player name formats:

- DK may include suffixes like `Jr.`, `III`, or hyphens: `"De'Von Achane"`
- nflreadpy uses `player_display_name` from the NFL API which normalises differently

`_norm()` strips punctuation, lowercases, and removes generational suffixes so that
`difflib.get_close_matches` can match across both formats reliably.


In [ ]:
def _norm(name: str) -> str:
    """Lowercase + strip punctuation + remove generational suffixes."""
    name = name.lower().strip()
    name = re.sub(r"['\'\-\.,]", "", name)
    name = re.sub(r"\s+(jr|sr|ii|iii|iv|v)$", "", name)
    return re.sub(r"\s+", " ", name)


# Quick sanity check
assert _norm("De'Von Achane Jr.") == "devon achane"
assert _norm("Patrick Mahomes II") == "patrick mahomes"
print("Name normalisation OK")


## 3 · Projection & Salary Data Loading

### 3a — Our model projections

`load_projections()` reads the weekly CSV produced by `predict_fantasy.ipynb`.
These files live in `fantasy/fantasy_projections/projections_{season}_week{week:02d}.csv`
and contain our XGBoost-based `projected_pts` for every active skill-position player.

We use our model projections (rather than DK's season average) because:
- Our model incorporates current injury status, depth chart position, and matchup difficulty.
- DK's `AvgPointsPerGame` is a season-long average and doesn't react to week-specific context.
- In backtesting our model beats the 3-week rolling average baseline at every position.

### 3b — DraftKings salary export

`load_dk_salaries()` parses the CSV exported directly from the DK contest lobby.
The standard columns are `Position`, `Name`, `Salary`, `TeamAbbrev`, `AvgPointsPerGame`.
Salary may include a `$` prefix depending on the DK export version.

For **DST** (team defenses) we have no position-specific projection model, so we fall back to
DK's `AvgPointsPerGame` as a proxy. This is an area for future improvement.


In [ ]:
_DK_COL_MAP = {
    "Position":        "position",
    "Name":            "name",
    "Salary":          "salary",
    "TeamAbbrev":      "team",
    "AvgPointsPerGame":"avg_pts",
    "Game Info":       "game_info",
}


def available_weeks() -> list[tuple[int, int]]:
    """Return (season, week) tuples for existing projection CSVs, newest first."""
    files = sorted(PROJ_DIR.glob("projections_*_week*.csv"), reverse=True)
    out = []
    for f in files:
        m = re.match(r"projections_(\d{4})_week(\d{2})\.csv", f.name)
        if m:
            out.append((int(m.group(1)), int(m.group(2))))
    return out


def load_projections(season: int, week: int) -> pd.DataFrame:
    path = PROJ_DIR / f"projections_{season}_week{week:02d}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No projection file: {path}")
    df = pd.read_csv(path)
    df["_norm"] = df["player_display_name"].apply(_norm)
    return df


def load_dk_salaries(path_or_bytes) -> pd.DataFrame:
    """Accept a file path (str/Path) or raw bytes from an uploaded file."""
    if isinstance(path_or_bytes, (str, Path)):
        raw = open(path_or_bytes, "rb").read()
    else:
        raw = path_or_bytes

    df = pd.read_csv(io.BytesIO(raw))
    df = df.rename(columns={k: v for k, v in _DK_COL_MAP.items() if k in df.columns})

    df["salary"] = (
        df["salary"].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df["salary"]  = pd.to_numeric(df["salary"],  errors="coerce")
    df["avg_pts"] = pd.to_numeric(df.get("avg_pts", 0), errors="coerce").fillna(0)
    df = df.dropna(subset=["salary", "name", "position"])
    df["_norm"] = df["name"].apply(_norm)

    keep = ["position", "name", "salary", "team", "avg_pts", "game_info", "_norm"]
    return df[[c for c in keep if c in df.columns]].copy()


weeks = available_weeks()
print(f"Available projection weeks: {weeks[:5]}")


## 4 · Merging Projections with Salaries

`merge_projections()` links each DK player to our model projection via fuzzy name match
(cutoff 0.72 on the normalised strings).

**Match quality labels:**

| Label | Meaning |
|-------|---------|
| `model` | Matched to our XGBoost projection — preferred |
| `dk_avg` | No match found; falls back to DK season average |
| `dst` | Defense/Special Teams — always uses DK avg (no position model) |

Players labelled `dk_avg` are still included in the pool so the optimizer has enough
players to fill every slot. The pipeline notebook highlights them so you can review manually.

The **value** column is `proj_pts / (salary / 1000)` — points per $1,000 of salary.
Sorting by value surfaces the best bargain picks at each position.


In [ ]:
def merge_projections(dk_df: pd.DataFrame, proj_df: pd.DataFrame) -> pd.DataFrame:
    proj_norms  = proj_df["_norm"].tolist()
    proj_lookup = {r["_norm"]: r for _, r in proj_df.iterrows()}

    pts_list, match_list = [], []

    for _, row in dk_df.iterrows():
        if row["position"] == "DST":
            pts_list.append(row.get("avg_pts", 0))
            match_list.append("dst")
            continue

        hits = difflib.get_close_matches(row["_norm"], proj_norms, n=1, cutoff=0.72)
        if hits:
            pts_list.append(proj_lookup[hits[0]]["projected_pts"])
            match_list.append("model")
        else:
            pts_list.append(row.get("avg_pts", 0))
            match_list.append("dk_avg")

    result = dk_df.drop(columns=["_norm"], errors="ignore").copy()
    result["proj_pts"] = pts_list
    result["match"]    = match_list
    result["value"]    = (result["proj_pts"] / (result["salary"] / 1000)).round(2)
    return result.reset_index(drop=True)


## 5 · Integer Linear Programming Formulation

The DK Classic lineup problem is a **binary integer program**:

**Decision variables:**  $x_i \in \{0, 1\}$ for each player $i$ in the pool.

**Objective:** Maximise total projected points
$$\max \sum_i \text{proj\_pts}_i \cdot x_i$$

**Constraints:**

| Rule | Formula |
|------|---------|
| Salary cap | $\sum_i \text{salary}_i \cdot x_i \leq 50{,}000$ |
| Total players | $\sum_i x_i = 9$ |
| QB | $\sum_{i \in QB} x_i = 1$ |
| RB | $\sum_{i \in RB} x_i \geq 2$ |
| WR | $\sum_{i \in WR} x_i \geq 3$ |
| TE | $\sum_{i \in TE} x_i \geq 1$ |
| DST | $\sum_{i \in DST} x_i = 1$ |
| Team max | $\sum_{i \in \text{team } t} x_i \leq 8 \quad \forall t$ |

The **FLEX slot** is implicit: with 9 total players, 1 QB, 1 DST, and the position minimums
(2 RB + 3 WR + 1 TE = 6), one extra RB, WR, or TE is chosen automatically by the solver
to fill the 9th slot.

We use **PuLP** with the bundled **CBC** solver. For a pool of ~200 players this solves in
under a second. Lock/exclude constraints add equality/zero bounds directly to the LP.


In [ ]:
def _assign_slots(lineup_df: pd.DataFrame) -> pd.DataFrame:
    """Label each row with its DK roster slot (QB/RB/WR/TE/FLEX/DST)."""
    df    = lineup_df.copy().reset_index(drop=True)
    slots = [""] * len(df)
    seen  = {"RB": 0, "WR": 0, "TE": 0}
    limits = {"RB": 2, "WR": 3, "TE": 1}

    for i, row in df.iterrows():
        pos = row["position"]
        if pos in ("QB", "DST"):
            slots[i] = pos
        elif pos in seen:
            seen[pos] += 1
            slots[i] = pos if seen[pos] <= limits[pos] else "FLEX"

    df.insert(0, "Slot", slots)
    return df


def optimize_lineup(
    players: pd.DataFrame,
    budget: int = BUDGET,
    locked: list[str] | None = None,
    excluded: list[str] | None = None,
) -> pd.DataFrame | None:
    df      = players.reset_index(drop=True)
    locked  = set(locked or [])
    excluded = set(excluded or [])
    n       = len(df)

    prob = pulp.LpProblem("DFS_Classic", pulp.LpMaximize)
    x    = [pulp.LpVariable(f"x{i}", cat="Binary") for i in range(n)]

    def pidx(pos):
        return [i for i in range(n) if df.iloc[i]["position"] == pos]

    qbs, rbs, wrs, tes, dsts = (pidx(p) for p in ["QB","RB","WR","TE","DST"])

    # Objective
    prob += pulp.lpSum(df.iloc[i]["proj_pts"] * x[i] for i in range(n))

    # Constraints
    prob += pulp.lpSum(df.iloc[i]["salary"] * x[i] for i in range(n)) <= budget
    prob += pulp.lpSum(x[i] for i in range(n)) == 9
    prob += pulp.lpSum(x[i] for i in qbs)  == 1
    prob += pulp.lpSum(x[i] for i in dsts) == 1
    prob += pulp.lpSum(x[i] for i in rbs)  >= 2
    prob += pulp.lpSum(x[i] for i in wrs)  >= 3
    prob += pulp.lpSum(x[i] for i in tes)  >= 1

    for team in df["team"].dropna().unique():
        tidx = [i for i in range(n) if df.iloc[i]["team"] == team]
        if len(tidx) > 8:
            prob += pulp.lpSum(x[i] for i in tidx) <= 8

    for name in locked:
        for i in df[df["name"] == name].index:
            prob += x[i] == 1
    for name in excluded:
        for i in df[df["name"] == name].index:
            prob += x[i] == 0

    status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[status] != "Optimal":
        return None

    selected = [i for i in range(n) if pulp.value(x[i]) > 0.5]
    order    = {"QB":0,"RB":1,"WR":2,"TE":3,"DST":5}
    lineup   = df.iloc[selected].copy()
    lineup["_sort"] = lineup["position"].map(order).fillna(4)
    lineup   = lineup.sort_values("_sort").drop(columns=["_sort"])
    return _assign_slots(lineup)


print("Optimizer functions loaded.")


## 6 · Interpreting Results

**Value metric** (`proj_pts / (salary / 1000)`)
- A value of 4.0 means the player is projected to score 4 pts per $1k of salary.
- The optimizer maximises total points — not value directly — but high-value players
  naturally unlock salary for a strong anchor pick (e.g. a $9k QB).

**FLEX pick logic**
- The ILP chooses the FLEX player globally — it may add a 3rd RB, 4th WR, or 2nd TE
  depending on which option adds the most projected points within the remaining cap.
- You don't need to pre-specify the FLEX position.

**When the optimizer returns `None`**
- Not enough players at a position (e.g. only 1 QB in the salary CSV).
- Lock constraints leave no feasible solution within the cap.
- Usually fixed by removing a lock or checking the uploaded CSV for missing rows.

## 7 · Next Steps

The current optimizer selects a single optimal lineup. Future improvements:

1. **Multi-lineup generation** — produce N distinct lineups for GPP tournaments using
   ownership diversity constraints (force different FLEX picks across lineups).
2. **Correlation / game-stacking** — add constraints to include 2+ players from the same game,
   exploiting score correlation in shootouts.
3. **Ownership leverage** — weight by inverse projected ownership to differentiate from the field.
4. **DST projection model** — train a simple model on defensive matchup difficulty, implied
   team total, and home/away to replace the DK season-average fallback.
5. **Salary efficiency bands** — flag players whose salary has moved since the season average
   was set, indicating recency-priced information the model may not yet reflect.
